# Match-Level Stats from StatsBomb CSVs

This notebook loads StatsBomb Open Data event CSV files from `data/raw`,
computes simple match-level statistics (shots, passes, pressures, xG),
and produces summary tables for basic analysis.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


## Load all event CSVs

We assume each match's events are stored as a separate CSV under
`data/raw/**/events_*.csv` (e.g. created by the StatsBomb download script).


In [ ]:
csv_files = list(DATA_DIR.glob("raw/**/events_*.csv"))
len(csv_files), csv_files[:5]


In [ ]:
def load_events_from_csvs(files: list[Path]) -> pd.DataFrame:
    """Load and concatenate multiple StatsBomb event CSV files."""
    dfs: list[pd.DataFrame] = []
    for f in files:
        df = pd.read_csv(f)
        # Ensure match_id is present (it should be in the CSV); if not, infer from filename
        if "match_id" not in df.columns:
            try:
                match_id = int(f.stem.split("_")[-1])
                df["match_id"] = match_id
            except Exception:
                pass
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


events = load_events_from_csvs(csv_files)
events.head()


## Basic event-type overview

Look at how many events of each type we have.


In [ ]:
event_type_counts = events["type"].value_counts(dropna=False)
event_type_counts.head(20)


## Derive match-level statistics

We compute, per match and team:

- Shots and goals
- Sum of StatsBomb xG
- Passes and completed passes
- Pressures


In [ ]:
# Normalize type / outcome column names across different StatsBomb exports
type_col = "type" if "type" in events.columns else "type_name"
outcome_col = "shot_outcome" if "shot_outcome" in events.columns else "shot_outcome_name" if "shot_outcome_name" in events.columns else None

shots = events[events[type_col].str.contains("Shot", na=False)].copy()
passes = events[events[type_col].str.contains("Pass", na=False)].copy()
pressures = events[events[type_col].str.contains("Pressure", na=False)].copy()

# Identify goals
if outcome_col is not None:
    goals_mask = shots[outcome_col].fillna("").str.contains("Goal")
else:
    goals_mask = False

shots["is_goal"] = goals_mask.astype(int)

# Ensure xG column name
xg_col = "shot_statsbomb_xg" if "shot_statsbomb_xg" in shots.columns else None
if xg_col is None:
    shots["shot_statsbomb_xg"] = np.nan
    xg_col = "shot_statsbomb_xg"

group_cols = ["match_id", "team"] if "team" in events.columns else ["match_id", "team_name"]

shot_stats = (
    shots.groupby(group_cols)
    .agg(
        shots=(type_col, "size"),
        goals=("is_goal", "sum"),
        xg=(xg_col, "sum"),
    )
)

pass_stats = (
    passes.groupby(group_cols)
    .agg(
        passes=(type_col, "size"),
    )
)

if "pass_outcome" in passes.columns:
    completed_mask = passes["pass_outcome"].isna() | (passes["pass_outcome"] == "Complete")
    passes["completed"] = completed_mask.astype(int)
    pass_completed = passes.groupby(group_cols)["completed"].sum().rename("completed_passes")
    pass_stats = pass_stats.join(pass_completed, how="left")

pressure_stats = pressures.groupby(group_cols).size().rename("pressures")

match_team_stats = shot_stats.join(pass_stats, how="outer").join(pressure_stats, how="outer").fillna(0)
match_team_stats.reset_index().head(20)


## Inspect stats for a single match

Pick one match to verify the computed statistics.


In [ ]:
example_match_id = events["match_id"].iloc[0]
match_team_stats.reset_index().query("match_id == @example_match_id")
